In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]

def node_a(state: State):
    return {"foo": "a", "bar": ["a"]}

def node_b(state: State):
    return {"foo": "b", "bar": ["b"]}


workflow = StateGraph(State)
workflow.add_node(node_a)
workflow.add_node(node_b)
workflow.add_edge(START, "node_a")
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", END)

checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}
graph.invoke({"foo": "", "bar":[]}, config)

{'foo': 'b', 'bar': ['a', 'b']}

In [2]:
checkpointer


In [3]:
from langchain_core.runnables import RunnableConfig

def my_node(state: State, config: RunnableConfig):
    checkpoint_ns = config["configurable"]["checkpoint_ns"]
    # "" for the parent graph, "node_name:uuid" for a subgraph

In [4]:
# get the latest state snapshot
config = {"configurable": {"thread_id": "1"}}
graph.get_state(config)

# get a state snapshot for a specific checkpoint_id
config = {"configurable": {"thread_id": "1", "checkpoint_id": "1ef663ba-28fe-6528-8002-5a559208592c"}}
graph.get_state(config)

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1ef663ba-28fe-6528-8002-5a559208592c'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [5]:
config = {"configurable": {"thread_id": "1"}}
list(graph.get_state_history(config))

[StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f120efc-6692-6357-8002-83e6abbfc37a'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-16T04:22:42.324770+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f120efc-668f-6c4b-8001-2dde4e312611'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'foo': 'a', 'bar': ['a']}, next=('node_b',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f120efc-668f-6c4b-8001-2dde4e312611'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-16T04:22:42.323770+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f120efc-668f-6c4a-8000-730d986f85f1'}}, tasks=(PregelTask(id='8d1c5773-e2bf-2fa8-baa3-a166417594dc', name='node_b', path=('__pregel_pull', 'node_b'), error=None, interrupts

In [6]:
history = list(graph.get_state_history(config))

In [7]:
before_node_b = next(s for s in history if s.next == ("node_b",))

In [8]:
before_node_b

StateSnapshot(values={'foo': 'a', 'bar': ['a']}, next=('node_b',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f120efc-668f-6c4b-8001-2dde4e312611'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-16T04:22:42.323770+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f120efc-668f-6c4a-8000-730d986f85f1'}}, tasks=(PregelTask(id='8d1c5773-e2bf-2fa8-baa3-a166417594dc', name='node_b', path=('__pregel_pull', 'node_b'), error=None, interrupts=(), state=None, result={'foo': 'b', 'bar': ['b']}),), interrupts=())

In [9]:
step_2 = next(s for s in history if s.metadata["step"] == 2)

In [10]:
step_2

StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f120efc-6692-6357-8002-83e6abbfc37a'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-16T04:22:42.324770+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f120efc-668f-6c4b-8001-2dde4e312611'}}, tasks=(), interrupts=())

In [11]:
forks = [s for s in history if s.metadata["source"] == "update"]

In [12]:
forks

[]